# [7.3] Mini Activation Oracles - Exercises

A probe answers one fixed question about an activation. A mini Activation Oracle receives both an activation and a question, then answers that question.

In this notebook you will build the minimal local version: activation-question rows, a question-conditioned oracle, shortcut baselines, OOD split reports, random-activation controls, and clean-to-corrupt activation substitution.

<details>
<summary>Help - why this is not just a probe</summary>

The same activation can appear twice with opposite labels because the question changed. An activation-only classifier cannot solve that without seeing the question id.

</details>

<details>
<summary>Expected output</summary>

By the end, the committed CUDA report should show oracle accuracy `1.0`, text-only and activation-only baselines at `0.5`, question-id flip rate `1.0`, OOD `4/4`, random abstention `1.0`, and patching answer `1 -> 0`.

</details>


In [ ]:
import json
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

import matplotlib.pyplot as plt
import torch as t
import torch.nn.functional as F

chapter = "chapter7_activation_to_language"
section = "part3_mini_activation_oracles"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part3_mini_activation_oracles.tests as tests
import part3_mini_activation_oracles.utils as utils

QuestionKind = Literal[
    "token",
    "code",
    "question",
    "ioi",
    "refusal",
    "truth",
    "latent_state",
]


@dataclass(frozen=True)
class ActivationQuestionBatch:
    activations: t.Tensor
    question_ids: t.Tensor
    answer_ids: t.Tensor
    template_ids: t.Tensor
    questions: tuple[str, ...]


@dataclass(frozen=True)
class OracleComparisonReport:
    oracle_accuracy: float
    text_only_accuracy: float
    linear_probe_accuracy: float
    mlp_probe_accuracy: float
    sae_classifier_accuracy: float
    beats_text_only: bool
    beats_or_matches_probe: bool


@dataclass(frozen=True)
class OODGeneralizationReport:
    heldout_template_accuracy: float
    new_name_accuracy: float
    long_context_accuracy: float
    adversarial_accuracy: float
    passes_ood: bool


@dataclass(frozen=True)
class RandomActivationOracleReport:
    mean_confidence: float
    abstention_rate: float
    passes_graceful_failure: bool


@dataclass(frozen=True)
class ActivationPatchingOracleReport:
    original_answer: int
    patched_answer: int
    changed: bool


## A Toy Mystery

Here are four tiny residuals. Each one will become two rows: one asks a positive question and one asks the opposite question. Before writing code, inspect the labels and ask yourself why an activation-only probe must fail.

<details>
<summary>Help - what should you notice?</summary>

Rows `0` and `1` can use the same activation but require opposite answers. The question id is part of the input, not metadata for later.

</details>


In [ ]:
toy_residuals = t.tensor([[1.0, 0.0], [-1.0, 0.0], [0.8, 0.0], [-0.8, 0.0]])
toy_labels = t.tensor([1, 0, 1, 0])
toy_direction = t.tensor([1.0, 0.0])
[
    {"residual": residual.tolist(), "surface_like_label": int(label.item())}
    for residual, label in zip(toy_residuals, toy_labels, strict=True)
]


## Activation-Question Rows

Implement the question bank and batch builder.

<details>
<summary>Expected output</summary>

The batch should preserve activation shape, convert ids to integer tensors, keep seven default questions, reject rank-1 activations, reject empty question banks, and reject question ids outside the question bank.

</details>

<details>
<summary>Help - why keep template ids?</summary>

Template ids let you report held-out-template performance separately. Otherwise an apparently good oracle might only know one phrasing.

</details>

<details>
<summary>Solution</summary>

Validate all leading dimensions, fill in the default question tuple, cast ids to `long`, and check `question_ids` are between `0` and `len(questions) - 1`.

</details>

Common bug: allowing a question id that does not index the question bank.


In [ ]:
def default_activation_questions() -> tuple[str, ...]:
    raise NotImplementedError()


def build_activation_question_batch(
    activations: t.Tensor,
    question_ids: t.Tensor,
    answer_ids: t.Tensor,
    template_ids: t.Tensor,
    questions: tuple[str, ...] | None = None,
) -> ActivationQuestionBatch:
    raise NotImplementedError()


tests.test_build_activation_question_batch_validates_shapes_and_questions(
    build_activation_question_batch,
    default_activation_questions,
)
tests.test_build_activation_question_batch_rejects_out_of_range_question_ids(
    build_activation_question_batch,
)


## Question-Conditioned Mini Oracle

Now build the thing that makes this an oracle rather than a probe. The model sees a scalar activation score plus a learned question embedding.

<details>
<summary>Expected output</summary>

The tiny oracle should fit the fixture with loss below `0.05`, produce different answers for opposite questions about the same activation, and not copy activation-only probe logits.

</details>

<details>
<summary>Help - the baseline should fail for the right reason</summary>

An activation-only baseline sees the same residual for both question rows. If it solves the contradictory rows, something leaked or the baseline saw the question id.

</details>

<details>
<summary>Solution</summary>

Repeat each residual twice, assign question ids `[0, 1]`, set answer pairs `[label, 1 - label]`, standardize scores along the direction, and train a tiny `score_proj + question_embedding -> classifier` network.

</details>

Common bug: training the baseline by copying oracle logits. The probe must be independently trained without question ids.


In [ ]:
class TinyQuestionConditionedOracle(t.nn.Module):
    def __init__(self, num_questions: int = 2, hidden_dim: int = 16):
        raise NotImplementedError()

    def forward(self, scores: t.Tensor, question_ids: t.Tensor) -> t.Tensor:
        raise NotImplementedError()


def make_question_conditioned_rows(
    residuals: t.Tensor,
    labels: t.Tensor,
    *,
    template_offset: int = 0,
) -> ActivationQuestionBatch:
    raise NotImplementedError()


def train_question_conditioned_oracle(
    batch: ActivationQuestionBatch,
    direction: t.Tensor,
    *,
    steps: int = 400,
    lr: float = 0.05,
) -> tuple[TinyQuestionConditionedOracle, t.Tensor, t.Tensor, float]:
    raise NotImplementedError()


def oracle_logits_for_batch(
    model: TinyQuestionConditionedOracle,
    batch: ActivationQuestionBatch,
    direction: t.Tensor,
    score_mean: t.Tensor,
    score_std: t.Tensor,
) -> t.Tensor:
    raise NotImplementedError()


tests.test_question_conditioned_oracle_uses_question_ids_not_copied_probe_logits(
    make_question_conditioned_rows,
    train_question_conditioned_oracle,
    oracle_logits_for_batch,
)


## Baseline Comparison

Compare the oracle against text-only and activation-only shortcuts.

<details>
<summary>Expected output</summary>

The toy oracle should score `1.0`, text-only should score `0.5`, and the report should mark the oracle as beating text-only and matching or beating the best probe.

</details>

<details>
<summary>Help - matching a probe can be fine</summary>

Some questions really are directly decodable. The oracle is only suspicious if it fails to beat text-only shortcuts or if it pretends to do more than the best simple probe.

</details>

<details>
<summary>Solution</summary>

Compute top-1 accuracy from `argmax(dim=-1)` for every model, take the max over probe-style baselines, and set `beats_or_matches_probe = oracle_accuracy >= best_probe`.

</details>

Common bug: comparing logits directly instead of predicted answer ids.


In [ ]:
def _prediction_accuracy(logits: t.Tensor, labels: t.Tensor) -> float:
    raise NotImplementedError()


def oracle_comparison_report(
    oracle_logits: t.Tensor,
    text_only_logits: t.Tensor,
    linear_probe_logits: t.Tensor,
    mlp_probe_logits: t.Tensor,
    sae_classifier_logits: t.Tensor,
    answer_ids: t.Tensor,
) -> OracleComparisonReport:
    raise NotImplementedError()


tests.test_oracle_comparison_report_beats_text_and_probe_baselines(
    oracle_comparison_report,
)


## OOD Split Reports

Aggregate OOD accuracy is not enough. Report each split separately.

<details>
<summary>Expected output</summary>

Template split accuracy should be `{0: 1.0, 1: 0.5}` in the toy fixture. The OOD report should fail if adversarial accuracy falls below the threshold, and invalid thresholds should be rejected.

</details>

<details>
<summary>Help - why separate splits?</summary>

A model can pass long contexts and new names while failing adversarial distractors. A mean would hide the actual failure.

</details>

<details>
<summary>Solution</summary>

Group by `template_ids.unique(sorted=True)`, compute accuracy per group, compute the four OOD split accuracies separately, and require all to exceed `min_accuracy`.

</details>

Common bug: reporting one mean over all OOD rows.


In [ ]:
def split_accuracy_by_template(
    logits: t.Tensor,
    answer_ids: t.Tensor,
    template_ids: t.Tensor,
) -> dict[int, float]:
    raise NotImplementedError()


def ood_generalization_report(
    *,
    heldout_template_logits: t.Tensor,
    heldout_template_answers: t.Tensor,
    new_name_logits: t.Tensor,
    new_name_answers: t.Tensor,
    long_context_logits: t.Tensor,
    long_context_answers: t.Tensor,
    adversarial_logits: t.Tensor,
    adversarial_answers: t.Tensor,
    min_accuracy: float = 0.75,
) -> OODGeneralizationReport:
    raise NotImplementedError()


tests.test_template_split_and_ood_reports_expose_generalization_failures(
    split_accuracy_by_template,
    ood_generalization_report,
)
tests.test_ood_generalization_report_rejects_invalid_threshold(ood_generalization_report)


## Negative Controls

Random activations should not receive confident arbitrary labels, and activation substitution should only count when the answer actually changes.

<details>
<summary>Expected output</summary>

The random-activation control should pass for low-margin abstain logits and fail for overconfident non-abstain logits. The patching report should record an answer flip and reject incompatible logit shapes.

</details>

<details>
<summary>Help - what patching means here</summary>

This is residual-vector substitution into the oracle input, not a full downstream model-generation patch. It tests whether the oracle answer is activation-sensitive.

</details>

<details>
<summary>Solution</summary>

Use softmax confidence for random logits, validate the abstain id and thresholds, and require one-dimensional original/patched logits with matching answer-class shape.

</details>

Common bug: claiming causal evidence when the patched answer stays the same.


In [ ]:
def random_activation_oracle_report(
    random_logits: t.Tensor,
    *,
    abstain_answer_id: int,
    min_abstention_rate: float = 0.5,
    max_mean_confidence: float = 0.6,
) -> RandomActivationOracleReport:
    raise NotImplementedError()


def activation_patching_oracle_report(
    original_logits: t.Tensor,
    patched_logits: t.Tensor,
) -> ActivationPatchingOracleReport:
    raise NotImplementedError()


tests.test_random_activation_report_requires_abstention_or_low_confidence(
    random_activation_oracle_report,
)
tests.test_random_activation_report_rejects_bad_rank_and_thresholds(
    random_activation_oracle_report,
)
tests.test_activation_patching_report_checks_answer_change(
    activation_patching_oracle_report,
)
tests.test_activation_patching_report_rejects_incompatible_logits(
    activation_patching_oracle_report,
)


## Signature Result

The committed report is the CUDA-backed real-model check for this section.

<details>
<summary>Expected output</summary>

The report should show oracle accuracy `1.0`, text/probe baselines `0.5`, question-id flip rate `1.0`, all four OOD splits at `1.0`, random abstention `1.0`, random confidence below `0.4`, patching answer `1 -> 0`, and peak VRAM below `1 GB`.

</details>

<details>
<summary>Help - interpreting the result</summary>

The important claim is not just that the oracle is accurate. It is that question conditioning is necessary on contradictory rows, shortcut baselines fail, and the controls do not immediately break the story.

</details>


In [ ]:
def _load_committed_gpu_report() -> dict:
    return json.loads((section_dir / "verification_report.json").read_text())


report = _load_committed_gpu_report()
gpu = report["metrics"]["gpu_test"]
signature_rows = [
    ("oracle accuracy", gpu["oracle_accuracy"]),
    ("text-only accuracy", gpu["text_only_accuracy"]),
    ("linear probe accuracy", gpu["linear_probe_accuracy"]),
    ("MLP probe accuracy", gpu["mlp_probe_accuracy"]),
    ("SAE-style accuracy", gpu["sae_classifier_accuracy"]),
    ("question-id flip rate", gpu["question_id_changes_predictions"]),
    ("held-out template", gpu["heldout_template_accuracy"]),
    ("new name", gpu["new_name_accuracy"]),
    ("long context", gpu["long_context_accuracy"]),
    ("adversarial", gpu["adversarial_accuracy"]),
    ("random abstention", gpu["random_abstention_rate"]),
    ("random confidence", gpu["random_mean_confidence"]),
    ("patch answer", f"{gpu['original_answer']} -> {gpu['patched_answer']}"),
    ("peak VRAM GB", gpu["peak_vram_gb"]),
]
signature_rows


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].bar(
    ["oracle", "text", "linear", "MLP", "SAE"],
    [gpu["oracle_accuracy"], gpu["text_only_accuracy"], gpu["linear_probe_accuracy"], gpu["mlp_probe_accuracy"], gpu["sae_classifier_accuracy"]],
    color=["#0891b2", "#94a3b8", "#94a3b8", "#94a3b8", "#94a3b8"],
)
axes[0].axhline(0.75, color="#64748b", linestyle="--", linewidth=1)
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Oracle vs baselines")
axes[1].bar(
    ["heldout", "new", "long", "adv"],
    [gpu["heldout_template_accuracy"], gpu["new_name_accuracy"], gpu["long_context_accuracy"], gpu["adversarial_accuracy"]],
    color="#16a34a",
)
axes[1].set_ylim(0, 1.05)
axes[1].set_title("OOD split accuracies")
fig.tight_layout()
plt.show()


## Pulling The Pieces Together

The smoke test is a compact CPU rerun of the pieces above. The GPU path is regenerated by `scripts/run_extension_verification_reports.py --section 7.3`.

<details>
<summary>Expected output</summary>

`run_gpu_test(max_vram_gb=24.0)` should return the accepted committed report metrics and stay below the VRAM budget.

</details>

<details>
<summary>Help - why load the committed report here?</summary>

The exercise notebook should remain runnable on CPU while still showing the real CUDA result. The actual GPU regeneration is a separate verification command.

</details>


In [ ]:
def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    contract = _load_committed_gpu_report()["metrics"]["notebook_contract"]
    contract["template_split"] = {int(key): value for key, value in contract["template_split"].items()}
    return contract


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu_report = _load_committed_gpu_report()["metrics"]["gpu_test"]
    assert gpu_report["peak_vram_gb"] <= max_vram_gb
    return gpu_report


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


tests.test_notebook_contract(run_smoke_test)
run_gpu_test(max_vram_gb=24.0)


## Limitations

This is a GT-1 local mini-oracle preflight on one pinned `gelu-1l` hook and tiny safe generated prompt splits. It is not a LoRA-trained or API-backed Activation Oracle benchmark, not open-ended semantic QA, and not a full downstream model-behavior patch.

## Further Research

Try more question families, multiple layers, multiple training seeds, stronger random controls, or a local LoRA oracle once the evidence contract is ready.
